# 构建时点一致财务面板

## 目标

对每个公司、报告期、指标和单位，保留截止时点可用的最新披露版本，以及来源和可得时间。

本文件使用虚构教学数据，不是论文复现或生产数据。

## 准备

使用 Python 3.10+ 内核，按顺序运行全部单元格。计算仅依赖标准库，无需密钥、联网或额外数据文件。可在已有的 Jupyter 环境中打开。

输入已内嵌，与同目录 inputs.json 内容一致；可在下一个单元格中修改 args 试验。时间与单位必须显式保留。

In [ ]:
import json

# Synthetic inputs; no credentials or network access.
bundle = json.loads("{\"version\":1,\"tutorial\":\"pit-fundamentals-panel\",\"identity\":\"synthetic\",\"args\":[[{\"entity\":\"DEMO\",\"period\":\"2024-12-31\",\"metric\":\"revenue\",\"unit\":\"CNY_million\",\"value\":100,\"version\":\"v1\",\"publishedAt\":\"2025-03-20T18:00:00+08:00\",\"firstSeenAt\":\"2025-03-20T18:05:00+08:00\"},{\"entity\":\"DEMO\",\"period\":\"2024-12-31\",\"metric\":\"revenue\",\"unit\":\"CNY_million\",\"value\":105,\"version\":\"v2\",\"publishedAt\":\"2025-04-10T18:00:00+08:00\",\"firstSeenAt\":\"2025-04-10T18:02:00+08:00\"}],\"2025-03-31T23:59:59+08:00\"],\"expected\":[{\"entity\":\"DEMO\",\"period\":\"2024-12-31\",\"metric\":\"revenue\",\"unit\":\"CNY_million\",\"value\":100,\"version\":\"v1\",\"publishedAt\":\"2025-03-20T18:00:00+08:00\",\"firstSeenAt\":\"2025-03-20T18:05:00+08:00\",\"availableAt\":\"2025-03-20T10:05:00.000Z\"}]}")
args = bundle["args"]
expected = bundle["expected"]
print(json.dumps(args, ensure_ascii=False, indent=2))

## 步骤

### 1. 区分四种时间

报告期末描述经济期间；披露时间描述发布；首次观察描述你的系统何时拿到数据；研究截止时间描述可使用的信息边界。示例使用 max(publishedAt, firstSeenAt) 作为本数据链的可得时点，不把后来补采的文件假装成当时已持有。

### 2. 保留版本，而不是覆盖

把公司、报告期、指标、单位和版本作为记录身份。示例中同一报告期先披露100，后来修订为105。两条都保留；同版本不同内容应隔离核对。实际数据还要区分合并与母公司口径、累计与单季、会计准则及币种。

### 3. 先过滤可得性，再选版本

先排除可得时间晚于截止时点的记录，再从每组中选择披露时间最新的版本。若披露顺序相同但版本不同，示例拒绝猜测。不能先对全库取最新版本，再用报告期末判断是否属于过去。

### 4. 检查尚未披露与修订

把截止时间放到首次披露之前，应没有结果；放到两次披露之间，应得到100；放到修订可得之后，应得到105。若来源只有日期，没有可信发布时间，不能自动补成当天零点；应明确采用保守的下一交易日约定或标记待核对。

### 方法与假设

- 缺少历史版本或可信披露时间时，不能仅靠程序制造PIT数据。
- 示例函数在每个报告期内选版本，不负责选择最新报告期或计算TTM。
- 缺失记录不是0；真实可得性与许可须独立确认。

In [ ]:
def select_as_of(rows, cutoff):
    """Select the latest published eligible version, without overwriting inputs."""
    from datetime import datetime, timezone
    import math
    import re

    def parse(value):
        if not isinstance(value, str) or not re.search(r"T.*(Z|[+-]\d{2}:\d{2})$", value):
            raise ValueError("timezone_required")
        try:
            return datetime.fromisoformat(value.replace("Z", "+00:00"))
        except ValueError:
            raise ValueError("timezone_required") from None

    boundary = parse(cutoff)
    eligible, identities, publication_orders = {}, set(), set()
    for row in rows:
        value = row.get("value")
        if (not all(row.get(field) for field in ("entity", "period", "metric", "unit"))
                or type(value) not in (int, float) or not math.isfinite(value)):
            raise ValueError("invalid_record")
        published, observed = parse(row.get("publishedAt")), parse(row.get("firstSeenAt"))
        key = tuple(row[field] for field in ("entity", "period", "metric", "unit"))
        identity = (key, row.get("version"))
        if not row.get("version") or identity in identities:
            raise ValueError("duplicate_or_missing_version")
        identities.add(identity)
        available = max(published, observed)
        if available > boundary:
            continue
        publication_order = (key, published)
        if publication_order in publication_orders:
            raise ValueError("ambiguous_publication_order")
        publication_orders.add(publication_order)
        if key not in eligible or published > eligible[key][0]:
            timestamp = available.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")
            eligible[key] = (published, dict(row, availableAt=timestamp))
    return [eligible[key][1] for key in sorted(eligible)]


### 运行小样本

示例截止2025-03-31，输出v1与数值100，排除4月的v2。此结果只验证示例规则，不证明供应商提供完整历史版本。

In [ ]:
result = select_as_of(*args)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 检查

将每一行与网页示例的预期输出比较。修改输入后，断言失败可能正是预期结果：先解释差异，不要直接删除验证。

In [ ]:
assert result == expected, "Output differs from the reference synthetic example"
assert bundle["identity"] == "synthetic"
print("通过：结果与网页虚构示例一致。")

## 下一步

真实数据须先通过已认证 GET /v1/catalog 核对权限、字段、schema_major、窗口与来源，再按实际合同映射。这里列出的是候选输入身份，不保证可用或历史完整。不要把 API as_of 当作历史财报版本。真实输入替换后须重新验证；不要沿用这份小样本的通过结论。

- `cn.dataset.income`
- `cn.dataset.cashflow`
- `cn.dataset.balancesheet`

### 参考资料

- [Tushare：财务数据入口](https://tushare.pro/document/2?doc_id=16)
- [Dechow与Dichev：应计估计误差](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=277231)

[返回教程](https://tradingdatas.com/recipes/pit-fundamentals-panel/)